<a href="https://colab.research.google.com/github/Multiomics-Analytics-Group/course_multi-omics_analysis/blob/main/proteomics/notebooks/01_proteomics_preprocessing_quantmsdiann.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 Preprocessing Proteomics with quantms / DIA-NN

In this session we turn **raw mass spectrometer output** into a **table of numbers**
ready for downstream analysis. That single step is where most of the
domain-specific knowledge of proteomics lives, and it is also the step that is most
often treated as a black box.

We will use:
[**quantmsdiann**](https://quantmsdiann.quantms.org/), a
[Nextflow](https://www.nextflow.io/) workflow that runs
[DIA-NN](https://github.com/vdemichev/DiaNN) in a reproducible, portable way.

### What you will be able to do afterwards

1. Explain what happens between an injected serum sample and a
   protein intensity table.
2. Read and write an **SDRF** file, the standard that describes *which raw file is
   which sample*.
3. Launch an nf-core-style pipeline from a notebook and find your way around its
   results folder.
4. Load the protein matrix of our course dataset and judge whether it is fit for
   analysis.

## 🧬 The Dataset

Everything we do over the three days follows **one cohort of patients**.

**Sepsis** is a life-threatening organ dysfunction caused by a dysregulated *host*
response to infection. *Klebsiella pneumoniae* (KP) is one of the most common
Gram-negative causes. Some KP strains have acquired resistance to **carbapenems**, the
antibiotics of last resort, these are called **CRKP** (carbapenem-**r**esistant KP), as
opposed to **CSKP** (carbapenem-**s**usceptible KP). CRKP sepsis causes the death of 20–40 % of
patients, about twice the rate of CSKP sepsis.

Here is the clinical problem that motivates the data:

> A patient arrives in the emergency department with sepsis. Blood cultures and
> antibiotic-susceptibility testing take **1–3 days**. Treatment decisions cannot wait
> that long. Could a **blood test measuring the host's own molecules** tell us, on day 0,
> whether we are dealing with a resistant organism?

To answer that, He *et al.* (2026) profiled serum from three groups of septic patients:

| Group | n | What it is |
|---|---|---|
| **Con** | 15 | Sepsis with **negative** microbiological cultures |
| **CSKP** | 15 | Sepsis with confirmed **carbapenem-susceptible** *K. pneumoniae* |
| **CRKP** | 15 | Sepsis with confirmed **carbapenem-resistant** *K. pneumoniae* |

Note that **all three groups are septic patients**. The comparison is not
"sick vs. healthy", it is "which pathogen, and how resistant". That makes the expected
effect sizes small and the analysis challenging, which is exactly what real
biomedical data look like.

Every serum sample was measured on **two platforms**:

- **Proteomics** — which proteins, and how much of each (this notebook, and Day 2).
- **Metabolomics** — which small molecules, and how much of each (Day 1 afternoon, Day 3).

Because the *same* 45 patients were measured twice, we can integrate the two layers on
Day 3.

### The data

| | Proteomics | Metabolomics |
|---|---|---|
| Repository | [PXD075261](https://proteomecentral.proteomexchange.org/) (ProteomeXchange / iProX) | [MTBLS14016](https://www.ebi.ac.uk/metabolights/MTBLS14016) (MetaboLights) |
| Instrument | timsTOF Pro (Bruker) | QTRAP 6500 (SCIEX) |
| Acquisition | **diaPASEF** (data-independent) | MRM (widely-targeted) |
| Samples | 45 patients + 3 pooled QC | 45 patients + 6 pooled QC |
| Features | 1 458 protein groups | 1 073 annotated metabolites |

**Reference.** He J, Luo S, Xu W, *et al.* *Serum proteomic profiling of sepsis patients
reveals a protein-based diagnostic model, with metabolomic insights into
carbapenem-resistant Klebsiella pneumoniae infection.* Front Immunol. 2026;17:1818068.
[doi:10.3389/fimmu.2026.1818068](https://doi.org/10.3389/fimmu.2026.1818068)

## ⚙️ Proteomics Workflow

### 1. Sample preparation

Serum is dominated by a handful of very abundant proteins (albumin, immunoglobulins) —
they can make up 90 % of the protein mass and hide everything interesting. In this study
the authors used **superparamagnetic nanoparticle beads** to bind and enrich a broader
slice of the proteome, then **reduced** the disulfide bonds (TCEP), **alkylated** the free
cysteines (CAA), and **digested** the proteins into peptides with **trypsin**.

### 2. Liquid chromatography

The peptide mixture is pushed through a very thin column (75 µm × 15 cm) at 300 nL/min.
Peptides stick to the column and come off at different times depending on how hydrophobic
they are. This spreads tens of thousands of peptides out over a ~1 hour gradient so the
mass spectrometer sees only a manageable subset at any instant. The time at which a
peptide comes off is its **retention time (RT)**.

### 3. Mass spectrometry: DDA vs DIA

The instrument alternates between two kinds of measurement:

- **MS1** — a survey scan: "what masses are present right now?"
- **MS2** — a fragmentation scan: peptides are smashed into pieces, and the fragment
  masses act like a fingerprint that identifies the peptide sequence.

The crucial design choice is *how* to pick what to fragment:

| | **DDA** (data-**dependent**) | **DIA** (data-**independent**) |
|---|---|---|
| Strategy | Look at MS1, pick the *N* most intense peaks, fragment those | Ignore MS1; fragment *everything* in a series of wide *m/z* windows |
| Result | Clean, one-peptide-at-a-time MS2 spectra | Complex MS2 spectra containing many peptides at once |
| Missing values | Many — a peptide is only measured if it happened to be picked | Few — every mass range is fragmented in every cycle |
| Analysis difficulty | Easier (one spectrum, one peptide) | Harder (must computationally untangle mixtures) |

Our dataset is **DIA**, and specifically **diaPASEF** on a timsTOF instrument. PASEF adds
a third separation dimension: **ion mobility**. Ions are held in a trapped-ion-mobility
device and released according to their shape-to-charge ratio (reported as 1/K₀, here
0.6–1.6 V·s/cm²). Because peptide mobility correlates with mass, the DIA windows can be
tilted along the *m/z*–mobility plane, so almost every fragmentation event lands on real
peptides instead of empty space. In practice this means **more identifications and fewer
missing values** than classical DIA — at the price of much larger files and a harder
computational problem.

### 4. Turning spectra into proteins: DIA-NN

[DIA-NN](https://github.com/vdemichev/DiaNN) is the software that does the untangling. In
**library-free** mode (used here) it:

1. Digests the FASTA protein database *in silico* into all possible peptides.
2. **Predicts**, with deep learning, each peptide's retention time, ion mobility and
   fragmentation pattern.
3. Searches those predictions against the measured DIA spectra.
4. Controls the error rate with a **false discovery rate (FDR)** threshold — here 1 %.
   (Roughly: repeat the search against decoy sequences that cannot be real, and use the
   decoy hit rate to calibrate how many of your "real" hits are noise.)
5. Quantifies each protein with **MaxLFQ**, which combines the peptide signals belonging
   to a protein in a way that is robust to peptides that are missing in some runs.

The output is what we are after: a matrix of **protein groups × samples** filled with
intensities.

## 🧰 Nextflow

| Problem | What Nextflow does |
|---|---|
| "It works on my laptop" | Every step runs inside a **container** with pinned software versions |
| "Which parameters did we use?" | Parameters are declared in config files that live in version control |
| "It crashed after 6 hours" | `-resume` restarts from the last completed step |
| "We now have 500 samples" | The same command runs on a laptop, an HPC cluster or the cloud |

[**nf-core**](https://nf-co.re/) is a community of curated Nextflow pipelines with shared
conventions. `quantmsdiann` is built to nf-core standards by the
[bigbio](https://github.com/bigbio) group at EMBL-EBI, and is the DIA-NN branch of the
[quantms](https://docs.quantms.org/) family. Tomorrow's metabolomics session uses
`nf-core/metaboigniter` in exactly the same way — once you have learned one, you have
learned them all.

### What quantmsdiann does, step by step

1. **Input validation** — parse and check the SDRF metadata.
2. **File preparation** — convert/index the MS files (`.raw` → `.mzML`; Bruker `.d` is
   read natively by DIA-NN).
3. **In-silico spectral library** — deep-learning prediction from the FASTA.
4. **Preliminary analysis** — per-file calibration of mass accuracy and RT.
5. **Empirical library assembly** — build a consensus library from what was actually seen.
6. **Individual analysis** — search each file against the empirical library (in parallel).
7. **Final quantification** — protein / peptide / gene matrices, cross-run normalisation.
8. **MSstats conversion** — a long-format table ready for statistical modelling.
9. **Quality control** — an interactive [pmultiqc](https://github.com/bigbio/pmultiqc) report.

Steps 4–6 are the "two-pass" trick that makes library-free DIA work: a predicted library
is good enough to find peptides, but a library rebuilt from your *own* measurements is
far more accurate for quantification.

## 💻 Running QuantMS

We need three things: **Java** (Nextflow runs on the JVM), **Nextflow** itself, and a
**container engine** to execute the pipeline's tools.

In [ ]:
# Where the course files live. Change BRANCH if you are working on a fork.
COURSE_REPO = "Multiomics-Analytics-Group/course_multi-omics_analysis"
BRANCH = "main"
BASE_URL = f"https://raw.githubusercontent.com/{COURSE_REPO}/{BRANCH}"

# Working folders inside the Colab session.
WORKDIR = "proteomics"
print("Base URL for course files:", BASE_URL)

In [ ]:
# ! mkdir -p {WORKDIR}/data {WORKDIR}/database {WORKDIR}/results
!mkdir -p proteomics/data proteomics/database proteomics/results
!ls -d proteomics/*

### Java

Nextflow needs Java 17 or newer. On Colab we install it with `apt`. This takes a minute
or two — the output is long and can be ignored unless it ends in an error.

In [ ]:
!apt-get -qq update > /dev/null
!apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
!java -version

### Nextflow

The installer is a single shell script. We move the resulting binary somewhere on the
`PATH` so we can call `nextflow` from any cell.

In [ ]:
!wget -qO- https://get.nextflow.io | bash
!mv -f nextflow /usr/local/bin/nextflow && chmod +x /usr/local/bin/nextflow
!nextflow -v

### A container engine

`quantmsdiann` runs every tool inside a container — **Conda is not supported**, because
DIA-NN is not distributed as a Conda package. So we need one of:

- **Docker** — the default on a laptop, workstation or most HPC-adjacent servers.
- **Singularity / Apptainer** — the usual choice on HPC clusters, and our best option in
  Colab, which has no Docker daemon.

The cell below detects what is available and installs Apptainer if necessary. It sets
`CONTAINER_PROFILE`, which we then pass to Nextflow with `-profile`.

> ⚠️ **Warning.** Container support inside Google Colab depends on the runtime you
> are given, and Apptainer occasionally fails there for reasons outside our control. If
> the pipeline run below does not start, **that is not your mistake** — skip to the
> section *"The output we would have produced"*, which uses the study's real DIA-NN
> results. Everything after that point works regardless. If you have Docker on your own
> machine, the same commands run there unchanged.

In [ ]:
import shutil
import subprocess


def have_working_docker() -> bool:
    if shutil.which("docker") is None:
        return False
    return subprocess.run(["docker", "info"], capture_output=True).returncode == 0


if have_working_docker():
    CONTAINER_PROFILE = "docker"
elif shutil.which("apptainer") or shutil.which("singularity"):
    CONTAINER_PROFILE = "apptainer" if shutil.which("apptainer") else "singularity"
else:
    print("No container engine found — installing Apptainer ...")
    subprocess.run("apt-get -qq install -y software-properties-common > /dev/null", shell=True)
    subprocess.run("add-apt-repository -y ppa:apptainer/ppa > /dev/null 2>&1", shell=True)
    subprocess.run("apt-get -qq update > /dev/null", shell=True)
    subprocess.run("apt-get -qq install -y apptainer > /dev/null", shell=True)
    CONTAINER_PROFILE = "apptainer" if shutil.which("apptainer") else None

print("Container profile:", CONTAINER_PROFILE)
if CONTAINER_PROFILE is None:
    print(
        "\nNo container engine could be installed in this runtime.\n"
        "Read through the pipeline section, then continue from "
        "'The output we would have produced'."
    )

## 📋 The metadata: SDRF

Before any pipeline can run, it has to know **which file is which sample**. Getting this
wrong is the single most common cause of a wrong result in omics — and it never produces
an error message, only a confidently wrong answer.

The proteomics community standard for this is
[**SDRF**](https://github.com/bigbio/proteomics-metadata-standard)
(*Sample and Data Relationship Format*): one row per MS run, with three families of
columns.

| Column family | Meaning | Example |
|---|---|---|
| `characteristics[...]` | properties of the **sample** | `characteristics[organism part]` = `blood serum` |
| `comment[...]` | properties of the **measurement** | `comment[instrument]` = `NT=timsTOF Pro` |
| `factor value[...]` | the **experimental variable** you want to compare | `factor value[disease]` = `carbapenem-resistant ... sepsis` |

Values often use the `NT=...;AC=...` syntax: `NT` is the human-readable **N**ame **T**erm,
`AC` the **AC**cession in a controlled vocabulary (here mostly
[PSI-MS](https://www.ebi.ac.uk/ols4/ontologies/ms) and
[Unimod](https://www.unimod.org/)). Using accessions instead of free text is what lets a
machine — and a colleague in five years — know exactly what you meant.

In [ ]:
import pandas as pd

sdrf_url = f"{BASE_URL}/proteomics/data/PXD075261.sdrf.tsv"
sdrf = pd.read_csv(sdrf_url, sep="\t")
print(f"{sdrf.shape[0]} runs described by {sdrf.shape[1]} metadata columns")
sdrf.head()

Wide tables are hard to read. Let us look at one run transposed, which is also how you
should sanity-check your own SDRF before launching anything.

In [ ]:
sdrf.iloc[0].to_frame("run 1").rename_axis("SDRF column")

And the experimental design in one line:

In [ ]:
sdrf["factor value[disease]"].value_counts().rename("number of runs").to_frame()

### 💡 A real-world metadata trap

Look closely at the file names. The acquisition computer wrote the carbapenem-**resistant**
samples as `CPKP`, while the repository, the paper and the metabolomics submission all use
`CRKP`. The DIA-NN output we will analyse tomorrow still carries the `CPKP` spelling in
its column headers.

Nothing about this is unusual, and nothing warns you about it. Somebody has to notice and
harmonise the identifiers — otherwise the proteomics and metabolomics tables silently
fail to join, and 15 patients disappear from the multi-omics analysis. In this course that
harmonisation is done in
[`bin/build_curated_data.py`](https://github.com/Multiomics-Analytics-Group/course_multi-omics_analysis/blob/main/bin/build_curated_data.py).

In [ ]:
sdrf.loc[sdrf["source name"].str.startswith("CRKP"), ["source name", "comment[data file]"]].head(3)

### 🔍 Reading the acquisition order out of the file names

File names in proteomics are rarely arbitrary. These encode the plate well (`GD6`, `GE3`,
…) and a sequential run identifier (`50906`, `50908`, …) assigned by the instrument. Since
the identifier increases with time, we can reconstruct **the order in which the samples
were injected** — information the paper does not state anywhere, and which the SDRF
carries only implicitly.

It is worth ten lines of code to look.

In [ ]:
import re

runs = (
    sdrf["comment[data file]"]
    .str.extract(r"25062408_HJ_(?P<sample>[A-Za-z]+\d+)_(?P<well>[A-Z]{2}\d+)_1_(?P<run_id>\d+)\.d")
    .astype({"run_id": int})
    .assign(block=lambda d: d["sample"].str.replace(r"\d+$", "", regex=True))
    .sort_values("run_id")
)
print(" -> ".join(runs["block"].tolist()))
runs.groupby("block")["run_id"].agg(["min", "max", "count"]).sort_values("min")

⚠️ **Look at what that says.** All 15 control samples were run first, then all 15 CSKP, then
all 15 CRKP — and in the full run list the three pooled QC injections sit exactly at the
block boundaries. **Acquisition order is perfectly confounded with the experimental group.**

Why this matters, in one sentence: any instrument drift over the batch — sensitivity
decaying, column ageing, source contamination — will produce differences between the groups
that have nothing to do with biology, and no statistical test can separate the two, because
the design gives it no information to do so.

The fix, had anyone been asked at the design stage, is **randomising the injection order**,
or at minimum interleaving the groups. It costs nothing and it is not recoverable
afterwards.

This does not mean the published results are wrong. The three QC injections bracket the
blocks and can be used to bound the drift; the effects the authors report are moderate; and
they validated their key proteins by targeted PRM in an **independent** cohort, which is
exactly the right answer to this objection. But it does mean that every group difference we
find over the next two days carries an alternative explanation we cannot fully exclude, and
we will say so when it matters — on Day 3 in particular, where a classifier will look
suspiciously good.

> 🧬 The general lesson: **read your file names.** Metadata that nobody wrote down is often
> still recoverable, and the most consequential features of an experiment are frequently the
> ones nobody mentioned.

### ✋ Exercise 1 — read the acquisition metadata

Using the transposed view above, answer:

1. Which protease was used, and which modification was set as **fixed** rather than
   variable? Why is carbamidomethylation on cysteine always fixed?
2. What precursor mass tolerance was used? Would 500 ppm be a sensible alternative?
3. `comment[technical replicate]` is 1 for every run. What does that tell you about how
   much of the observed variability is instrumental versus biological?

## 🗄️ The sequence database

A library-free DIA search needs a **FASTA** file of candidate protein sequences — the
search space. The study used the human UniProt reference proteome (release 2025-01-13),
about 20 000 reviewed sequences and ~80 MB with isoforms.

We do **not** download it for the demonstration run below (the test profile brings its
own small database), but this is how you would get it. Note the two decisions hidden in
that URL: whether to include **isoforms**, and whether to restrict to **reviewed**
(Swiss-Prot) entries. A bigger search space means more candidate peptides, a harsher FDR
correction, and usually *fewer* confident identifications.

In [ ]:
UNIPROT_HUMAN = (
    "https://rest.uniprot.org/uniprotkb/stream"
    "?query=%28proteome%3AUP000005640%29+AND+%28reviewed%3Atrue%29"
    "&format=fasta&compressed=true"
)
print("To fetch the human reference proteome, run:\n")
print(f"!wget -q '{UNIPROT_HUMAN}' -O proteomics/database/uniprot_human.fasta.gz")
print("!gunzip -f proteomics/database/uniprot_human.fasta.gz")

> 🧬 In a real analysis you would also append a **contaminant** database (keratins from
> skin and hair, trypsin itself, serum albumin from lab reagents). Peptides that match
> contaminants are then removed *after* the search — you must let the search engine see
> them, otherwise their spectra get misassigned to real proteins.

## 🚀 Running the pipeline on a reduced dataset

Our 45 raw files are ~2.4 GB **each** — over 100 GB, and many CPU-hours. That is a real
constraint of proteomics, not an artefact of this course, and it is exactly why pipelines
are built to run on clusters.

For the hands-on we therefore use the pipeline's own **test profile**, which points at a
tiny public diaPASEF dataset ([PXD065380](https://www.ebi.ac.uk/pride/archive/projects/PXD065380),
human urine, timsTOF Pro 2) with a matching small FASTA. Same pipeline, same code path,
same output files — just small enough to finish while we watch.

```bash
nextflow run bigbio/quantmsdiann -r 2.3.0 \
    -profile test_dia_dotd,<docker|apptainer|singularity> \
    --outdir proteomics/results/test_run
```

Anatomy of that command:

| Part | Meaning |
|---|---|
| `nextflow run bigbio/quantmsdiann` | fetch and run the pipeline straight from GitHub |
| `-r 2.3.0` | pin the pipeline **version** — always do this in real projects |
| `-profile test_dia_dotd` | a bundle of preset parameters (input, database, thresholds) |
| `,apptainer` | how to execute the tools; profiles are comma-separated |
| `--outdir` | where results go (single dash = Nextflow option, double dash = pipeline parameter) |

> ⏱️ Expect **15–40 minutes**, most of it spent downloading container images and the
> deep-learning model. The log prints one line per process as it completes. Add
> `-resume` to any re-run to reuse finished steps.

In [ ]:
!nextflow pull bigbio/quantmsdiann -r 2.3.0

In [ ]:
# This is the cell that does the work. If it fails, read the warning above and move on.
if CONTAINER_PROFILE:
    command = (
        "nextflow run bigbio/quantmsdiann -r 2.3.0 "
        f"-profile test_dia_dotd,{CONTAINER_PROFILE} "
        "--outdir proteomics/results/test_run -resume"
    )
    print(command, "\n")
    !{command}
else:
    print("Skipped: no container engine available in this runtime.")

### Finding your way around the results

Nextflow writes two things: a `work/` directory (one hashed subfolder per task — this is
what `-resume` reads, and what you delete when you run out of disk) and the tidy
`--outdir` you asked for.

In [ ]:
!find proteomics/results/test_run -maxdepth 2 -type d 2>/dev/null | sort | head -30

In [ ]:
!find proteomics/results/test_run -type f \( -name "*.tsv" -o -name "*.parquet" -o -name "*.html" \) 2>/dev/null | sort | head -30

The files that matter downstream:

| File | Contents |
|---|---|
| `report.pg_matrix.tsv` | **protein groups × runs**, MaxLFQ intensities — the table we analyse |
| `report.pr_matrix.tsv` | the same at **precursor** (peptide + charge) level |
| `report.tsv` / `.parquet` | one row per identified precursor per run: scores, RT, FDR |
| `*_msstats_in.csv` | long format for statistical modelling with MSstats |
| `multiqc_report.html` | the QC report — **open this first**, every time |

> 💡 A good habit: before looking at a single fold-change, open the QC report and check
> the number of identifications per run, the retention-time stability and the
> total-ion-current. A run that failed technically will otherwise show up later as
> exciting biology.

## 🧾 The production command

For the real cohort, the command changes only in its inputs — this is the point of a
pipeline. We do **not** execute it here.

```bash
nextflow run bigbio/quantmsdiann -r 2.3.0 \
    -profile docker \
    --input  proteomics/data/PXD075261.sdrf.tsv \
    --database proteomics/database/uniprot_human.fasta \
    --outdir proteomics/results/PXD075261 \
    --root_folder proteomics/data/raw \
    --local_input_type d \
    --min_peptide_length 7 \
    --max_peptide_length 30 \
    --max_precursor_charge 4 \
    --allowed_missed_cleavages 2 \
    --mass_acc_ms1 15 --mass_acc_ms2 15 \
    -resume
```

- `--root_folder` / `--local_input_type` tell the pipeline to look for the files named in
  the SDRF **locally** instead of downloading them.
- The search parameters mirror the paper's Methods: tryptic peptides, up to 2 missed
  cleavages, 15 ppm tolerances, charges up to 4+.
- `.d.zip` archives are supported directly, so there is no need to unpack 100 GB.

### ✋ Exercise 2 — plan a run of your own

Take a dataset you actually work with, or invent a plausible one, and write down: how many
runs, which instrument, which acquisition method, which FASTA, and what the
`factor value[...]` column would contain. If you cannot fill in all five, you have found
the metadata you still need to collect.

## 📈 The output we would have produced

Below is the **real** `report.pg_matrix.tsv` from the study — DIA-NN 1.9.2, library-free,
MaxLFQ, 1 % FDR — exactly the file the production command produces. From here on, this
notebook works whether or not the pipeline ran.

We load a lightly curated copy in which the run-name columns have been replaced by clean
sample identifiers (`Con1`…`CRKP15`, `QC_pool1`…`3`).

In [ ]:
proteins = pd.read_csv(f"{BASE_URL}/proteomics/data/protein_groups_matrix.tsv", sep="\t")
metadata = pd.read_csv(f"{BASE_URL}/metadata/sample_metadata.tsv", sep="\t")

print(f"{proteins.shape[0]} protein groups x {proteins.shape[1] - 4} runs")
proteins.iloc[:5, :8]

The first four columns are annotation, the rest are intensities:

In [ ]:
annotation_cols = ["protein_group", "protein_names", "genes", "description"]
sample_cols = [c for c in proteins.columns if c not in annotation_cols]
qc_cols = [c for c in sample_cols if c.startswith("QC")]
patient_cols = [c for c in sample_cols if not c.startswith("QC")]

print("patient runs:", len(patient_cols))
print("pooled QC runs:", qc_cols)
proteins[annotation_cols].head()

In [ ]:
import matplotlib.pyplot as plt

intensities = proteins[patient_cols]
observed_per_protein = intensities.notna().sum(axis=1)

print(f"Overall missing values: {intensities.isna().to_numpy().mean():.1%}")
print(f"Proteins quantified in all {len(patient_cols)} patients: {(observed_per_protein == len(patient_cols)).sum()}")
print(f"Proteins quantified in at least 80% of patients: {(observed_per_protein >= 0.8 * len(patient_cols)).sum()}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
observed_per_protein.sort_values(ascending=False).reset_index(drop=True).plot(
    ax=axes[0], color="steelblue"
)
axes[0].set(
    xlabel="Protein groups (sorted)",
    ylabel="Number of patients quantified in",
    title="Data completeness per protein",
)
intensities.notna().sum(axis=0).plot(kind="bar", ax=axes[1], color="steelblue", width=0.85)
axes[1].set(
    xlabel="", ylabel="Protein groups quantified", title="Identifications per patient sample"
)
axes[1].tick_params(axis="x", labelsize=6, rotation=90)
fig.tight_layout()

The staircase on the left is the signature shape of any label-free experiment: a core of
proteins seen everywhere, a long tail seen occasionally. Where you cut that staircase is
a real analytical decision, and we will make it explicitly tomorrow.

The bars on the right are the run-level sanity check. A sample with far fewer
identifications than its neighbours is a technical outlier, and no amount of clever
statistics downstream will repair it.

In [ ]:
qc = proteins[qc_cols].dropna()
cv = (qc.std(axis=1) / qc.mean(axis=1)) * 100
print(f"{len(qc)} protein groups quantified in all three QC injections")
print(f"median technical CV: {cv.median():.1f}%")
ax = cv.clip(upper=100).plot(
    kind="hist", bins=40, color="steelblue", figsize=(7, 3.5),
    title="Technical variability across pooled QC injections",
)
ax.set_xlabel("Coefficient of variation (%)")

In [ ]:
import numpy as np

long = (
    intensities.melt(var_name="sample_id", value_name="intensity")
    .dropna()
    .merge(metadata[["sample_id", "group"]], on="sample_id")
)
long["log2_intensity"] = np.log2(long["intensity"])

fig, ax = plt.subplots(figsize=(7, 4))
order = ["Con", "CSKP", "CRKP"]
ax.boxplot([long.loc[long["group"] == g, "log2_intensity"] for g in order], showfliers=False)
ax.set_xticks(range(1, len(order) + 1))
ax.set_xticklabels(order)
ax.set(ylabel="log2 MaxLFQ intensity", title="Distribution of protein intensities per group")
fig.tight_layout()

The distributions overlap almost completely — as they should. The biology we are after is
not a global shift in the serum proteome; it is a change in **specific** proteins against
an unchanged background. Finding those, without being fooled by noise, is the subject of
Day 2.

## 📚 Further reading

- quantmsdiann documentation — <https://quantmsdiann.quantms.org/>
- Demichev *et al.* (2020) *DIA-NN: neural networks and interference correction enable
  deep proteome coverage in high throughput.* Nat Methods 17:41–44.
- Meier *et al.* (2020) *diaPASEF: parallel accumulation–serial fragmentation combined
  with data-independent acquisition.* Nat Methods 17:1229–1236.
- Dai *et al.* (2024) *quantms: a cloud-based pipeline for quantitative proteomics.*
  Nat Methods 21:1603–1607.
- Dai *et al.* (2021) *A proteomics sample metadata representation for multiomics
  integration and big data analysis (SDRF).* Nat Commun 12:5854.